# 03 - Pruning and Quantization (TensorFlow Lite)

This notebook uses the trained IDS model from `02_model_training.ipynb` and performs:

1. Baseline Keras evaluation
2. Magnitude-based pruning fine-tuning (target sparsity up to 0.5)
3. TFLite conversion (float + INT8 post-training quantization)
4. Accuracy, model size, and inference latency comparison

Outputs are saved under `artifacts/models/`.

In [7]:
import os
import time
import random
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow import keras
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score, confusion_matrix

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

BASE_DIR = os.path.abspath('..')
ARTIFACTS_DIR = os.path.join(BASE_DIR, 'artifacts')
MODEL_DIR = os.path.join(ARTIFACTS_DIR, 'models')
os.makedirs(MODEL_DIR, exist_ok=True)

def file_size_kb(path):
    return os.path.getsize(path) / 1024.0 if os.path.exists(path) else np.nan

print('TensorFlow:', tf.__version__)
print('Artifacts dir:', ARTIFACTS_DIR)
print('Model dir:', MODEL_DIR)

TensorFlow: 2.15.1
Artifacts dir: d:\SAG-Internship-Project\artifacts
Model dir: d:\SAG-Internship-Project\artifacts\models


In [8]:
X_train = np.load(os.path.join(ARTIFACTS_DIR, 'X_train.npy')).astype(np.float32)
y_train = np.load(os.path.join(ARTIFACTS_DIR, 'y_train.npy')).astype(np.int32)
X_test = np.load(os.path.join(ARTIFACTS_DIR, 'X_test.npy')).astype(np.float32)
y_test = np.load(os.path.join(ARTIFACTS_DIR, 'y_test.npy')).astype(np.int32)

# LSTM in notebook 02 uses sequence shape (timesteps, channels).
X_train_seq = X_train.reshape(X_train.shape[0], X_train.shape[1], 1)
X_test_seq = X_test.reshape(X_test.shape[0], X_test.shape[1], 1)

print('X_train_seq:', X_train_seq.shape)
print('X_test_seq :', X_test_seq.shape)
print('Attack rate (train):', float(y_train.mean()))
print('Attack rate (test) :', float(y_test.mean()))

X_train_seq: (82332, 36, 1)
X_test_seq : (175341, 36, 1)
Attack rate (train): 0.5506000097167566
Attack rate (test) : 0.6806223302022916


In [9]:
candidate_models = [
    os.path.join(MODEL_DIR, 'IDSmodel.h5'),
    os.path.join(MODEL_DIR, 'final_model.h5'),
    os.path.join(MODEL_DIR, 'final_model.keras'),
]

BASE_MODEL_PATH = next((p for p in candidate_models if os.path.exists(p)), None)
if BASE_MODEL_PATH is None:
    raise FileNotFoundError('No trained model found. Expected IDSmodel.h5 or final_model.* in artifacts/models/')

print('Using base model:', BASE_MODEL_PATH)

def load_model_compat(model_path):
    try:
        return keras.models.load_model(model_path)
    except Exception as e:
        print('Standard load_model failed:', e)
        if not model_path.endswith('.h5'):
            raise

        import json
        import h5py

        with h5py.File(model_path, 'r') as f:
            raw_cfg = f.attrs.get('model_config', None)
        if raw_cfg is None:
            raise

        if isinstance(raw_cfg, bytes):
            raw_cfg = raw_cfg.decode('utf-8')
        model_cfg = json.loads(raw_cfg)

        def _sanitize(obj):
            if isinstance(obj, dict):
                if obj.get('class_name') == 'InputLayer' and 'config' in obj:
                    cfg = obj['config']
                    if 'batch_shape' in cfg and 'batch_input_shape' not in cfg:
                        cfg['batch_input_shape'] = cfg.pop('batch_shape')
                    cfg.pop('optional', None)
                for k, v in list(obj.items()):
                    obj[k] = _sanitize(v)
                return obj
            if isinstance(obj, list):
                return [_sanitize(x) for x in obj]
            return obj

        model_cfg = _sanitize(model_cfg)
        model = keras.models.model_from_config(model_cfg)
        model.load_weights(model_path)
        return model

base_model = load_model_compat(BASE_MODEL_PATH)
base_loss, base_acc = base_model.evaluate(X_test_seq, y_test, verbose=0)
print(f'Baseline test accuracy: {base_acc*100:.2f}%')
print(f'Baseline model size: {file_size_kb(BASE_MODEL_PATH):.1f} KB')

Using base model: d:\SAG-Internship-Project\artifacts\models\final_model.h5
Standard load_model failed: Error when deserializing class 'InputLayer' using config={'batch_shape': [None, 36], 'dtype': 'float32', 'sparse': False, 'ragged': False, 'name': 'input_layer', 'optional': False}.

Exception encountered: Unrecognized keyword arguments: ['batch_shape', 'optional']



TypeError: Error when deserializing class 'Dense' using config={'name': 'dense_0', 'trainable': True, 'dtype': {'module': 'keras', 'class_name': 'DTypePolicy', 'config': {'name': 'float32'}, 'registered_name': None}, 'units': 64, 'activation': 'relu', 'use_bias': True, 'kernel_initializer': {'module': 'keras.initializers', 'class_name': 'GlorotUniform', 'config': {'seed': None}, 'registered_name': None}, 'bias_initializer': {'module': 'keras.initializers', 'class_name': 'Zeros', 'config': {}, 'registered_name': None}, 'kernel_regularizer': None, 'bias_regularizer': None, 'kernel_constraint': None, 'bias_constraint': None, 'quantization_config': None}.

Exception encountered: ('Keyword argument not understood:', 'quantization_config')

## Pruning Setup

If `tensorflow_model_optimization` is missing, install it first:

```python
!pip install -q tensorflow-model-optimization
```

In [ ]:
try:
    import tensorflow_model_optimization as tfmot
except ImportError as exc:
    raise ImportError(
        'Install tensorflow-model-optimization, then re-run this notebook.'
    ) from exc

batch_size = 256
epochs_prune = 5

end_step = int(np.ceil(X_train_seq.shape[0] / batch_size) * epochs_prune)
prune_schedule = tfmot.sparsity.keras.PolynomialDecay(
    initial_sparsity=0.0,
    final_sparsity=0.5,
    begin_step=0,
    end_step=end_step,
)

pruned_model = tfmot.sparsity.keras.prune_low_magnitude(
    base_model,
    pruning_schedule=prune_schedule,
)

pruned_model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-4),
    loss='binary_crossentropy',
    metrics=['accuracy'],
)

callbacks = [
    tfmot.sparsity.keras.UpdatePruningStep(),
    keras.callbacks.EarlyStopping(
        monitor='val_loss',
        patience=2,
        restore_best_weights=True,
        verbose=1,
    ),
]

In [ ]:
history = pruned_model.fit(
    X_train_seq,
    y_train,
    validation_split=0.2,
    epochs=epochs_prune,
    batch_size=batch_size,
    callbacks=callbacks,
    verbose=1,
)

stripped_model = tfmot.sparsity.keras.strip_pruning(pruned_model)
stripped_model.compile(loss='binary_crossentropy', metrics=['accuracy'])

pruned_h5_path = os.path.join(MODEL_DIR, 'IDSmodel_pruned.h5')
stripped_model.save(pruned_h5_path)

pruned_loss, pruned_acc = stripped_model.evaluate(X_test_seq, y_test, verbose=0)
print(f'Pruned model accuracy: {pruned_acc*100:.2f}%')
print(f'Pruned H5 size: {file_size_kb(pruned_h5_path):.1f} KB')

In [ ]:
def representative_data_gen():
    subset = X_train_seq[:2000]
    for i in range(subset.shape[0]):
        yield [subset[i:i+1].astype(np.float32)]

# Float TFLite from stripped pruned model
converter_fp = tf.lite.TFLiteConverter.from_keras_model(stripped_model)
tflite_fp = converter_fp.convert()
tflite_fp_path = os.path.join(MODEL_DIR, 'IDSmodel_float.tflite')
with open(tflite_fp_path, 'wb') as f:
    f.write(tflite_fp)

# INT8 PTQ
converter_int8 = tf.lite.TFLiteConverter.from_keras_model(stripped_model)
converter_int8.optimizations = [tf.lite.Optimize.DEFAULT]
converter_int8.representative_dataset = representative_data_gen
converter_int8.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS_INT8]
converter_int8.inference_input_type = tf.int8
converter_int8.inference_output_type = tf.int8
tflite_int8 = converter_int8.convert()

tflite_int8_path = os.path.join(MODEL_DIR, 'IDSmodel.tflite')
with open(tflite_int8_path, 'wb') as f:
    f.write(tflite_int8)

print(f'Float TFLite size: {file_size_kb(tflite_fp_path):.1f} KB')
print(f'INT8 TFLite size : {file_size_kb(tflite_int8_path):.1f} KB')

In [ ]:
def tflite_predict(tflite_path, x_data):
    interpreter = tf.lite.Interpreter(model_path=tflite_path)
    interpreter.allocate_tensors()
    input_details = interpreter.get_input_details()[0]
    output_details = interpreter.get_output_details()[0]

    in_scale, in_zero = input_details['quantization']
    out_scale, out_zero = output_details['quantization']
    preds = []

    for i in range(x_data.shape[0]):
        sample = x_data[i:i+1]
        if input_details['dtype'] == np.int8:
            sample = np.clip(np.round(sample / in_scale + in_zero), -128, 127).astype(np.int8)
        else:
            sample = sample.astype(np.float32)

        interpreter.set_tensor(input_details['index'], sample)
        interpreter.invoke()
        out = interpreter.get_tensor(output_details['index'])

        if output_details['dtype'] == np.int8:
            out = (out.astype(np.float32) - out_zero) * out_scale

        preds.append(out[0, 0])

    return np.array(preds, dtype=np.float32)

eval_n = min(5000, X_test_seq.shape[0])
x_eval = X_test_seq[:eval_n]
y_eval = y_test[:eval_n]

pred_fp = tflite_predict(tflite_fp_path, x_eval)
pred_int8 = tflite_predict(tflite_int8_path, x_eval)

acc_fp = accuracy_score(y_eval, (pred_fp >= 0.5).astype(np.int32))
acc_int8 = accuracy_score(y_eval, (pred_int8 >= 0.5).astype(np.int32))

print(f'Float TFLite accuracy ({eval_n} samples): {acc_fp*100:.2f}%')
print(f'INT8 TFLite accuracy  ({eval_n} samples): {acc_int8*100:.2f}%')

In [ ]:
def benchmark_tflite_ms(tflite_path, x_data, n=200):
    interpreter = tf.lite.Interpreter(model_path=tflite_path)
    interpreter.allocate_tensors()
    input_details = interpreter.get_input_details()[0]
    output_details = interpreter.get_output_details()[0]
    in_scale, in_zero = input_details['quantization']

    n = min(n, x_data.shape[0])
    start = time.perf_counter()
    for i in range(n):
        sample = x_data[i:i+1]
        if input_details['dtype'] == np.int8:
            sample = np.clip(np.round(sample / in_scale + in_zero), -128, 127).astype(np.int8)
        else:
            sample = sample.astype(np.float32)
        interpreter.set_tensor(input_details['index'], sample)
        interpreter.invoke()
        _ = interpreter.get_tensor(output_details['index'])
    elapsed = time.perf_counter() - start
    return (elapsed / n) * 1000.0

lat_fp = benchmark_tflite_ms(tflite_fp_path, X_test_seq, n=300)
lat_int8 = benchmark_tflite_ms(tflite_int8_path, X_test_seq, n=300)
print(f'Float TFLite latency: {lat_fp:.3f} ms/sample')
print(f'INT8 TFLite latency : {lat_int8:.3f} ms/sample')

In [ ]:
results = pd.DataFrame([
    {
        'model': 'Baseline Keras (.h5/.keras)',
        'accuracy': float(base_acc),
        'size_kb': file_size_kb(BASE_MODEL_PATH),
        'latency_ms': np.nan,
    },
    {
        'model': 'Pruned Keras (.h5)',
        'accuracy': float(pruned_acc),
        'size_kb': file_size_kb(pruned_h5_path),
        'latency_ms': np.nan,
    },
    {
        'model': 'Pruned Float TFLite',
        'accuracy': float(acc_fp),
        'size_kb': file_size_kb(tflite_fp_path),
        'latency_ms': float(lat_fp),
    },
    {
        'model': 'Pruned INT8 TFLite',
        'accuracy': float(acc_int8),
        'size_kb': file_size_kb(tflite_int8_path),
        'latency_ms': float(lat_int8),
    },
])

results['accuracy_pct'] = results['accuracy'] * 100.0
results = results[['model', 'accuracy_pct', 'size_kb', 'latency_ms']]
results = results.sort_values('size_kb').reset_index(drop=True)
results

results_path = os.path.join(ARTIFACTS_DIR, 'quantization_results.csv')
results.to_csv(results_path, index=False)
print('Saved:', results_path)
print('Saved:', tflite_int8_path)